## Setup

In [3]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

if not os.getcwd().endswith("etl"):
    os.chdir("./etl")

import requests
import pandas as pd
import pyspark
from io import BytesIO
from dotenv import load_dotenv
from json import load
from pyspark.sql import functions as F, Window
from tqdm import tqdm
from time import sleep

In [4]:
load_dotenv()

JDBC_PATH = os.getenv("JDBC_PATH")
_DB_URL = os.getenv("DB_URL")
_DB_NAME = os.getenv("DB_NAME")
_DB_USER = os.getenv("DB_USER")
_DB_PASSWORD = os.getenv("DB_PASSWORD")

if JDBC_PATH is None or _DB_URL is None or _DB_NAME is None or _DB_USER is None or _DB_PASSWORD is None:
    raise EnvironmentError("Set .env vars!")
if not os.path.exists(JDBC_PATH) or not os.path.isfile(JDBC_PATH):
    raise FileNotFoundError("JDBC driver not found!")

DB_URL = f"jdbc:mysql://{_DB_URL}/{_DB_NAME}"
DB_PROPERTIES = {"user": _DB_USER, "password": _DB_PASSWORD, "driver": "com.mysql.cj.jdbc.Driver"}

In [5]:
spark = pyspark.sql.SparkSession.builder.appName("ETL TCC").config("spark.jars", JDBC_PATH).getOrCreate()

26/04/21 21:36:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 21:36:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Extract

### Base bueiros

In [7]:
BUEIROS_URL = "https://www.der.sp.gov.br/WebSite/Arquivos/DadosAbertos/AtivosRodoviarios/Bueiros/Bueiros.xlsx"
res = requests.get(BUEIROS_URL)
res.raise_for_status()

In [8]:
content_wrapper = BytesIO(res.content)
pandas_df = pd.read_excel(content_wrapper)
df_raw_bueiros = spark.createDataFrame(pandas_df)

### Mocks

In [9]:
pandas_df = pd.read_csv("./mocks/distritos.csv").reset_index()
df_mock_distritos = spark.createDataFrame(pandas_df)

## Transform

#### 1. Distrito

In [81]:
df_transf_distritos = df_mock_distritos.withColumnRenamed("index", "id").withColumn("cidade", F.lit("São Paulo"))

#### 2. Coordenada

In [ ]:
df_raw_coords = df_transf_distritos.toPandas()
df_raw_coords["latitude"] = 0.0 
df_raw_coords["longitude"] = 0.0 


geolocator = Nominatim(user_agent="tcc")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

for i in tqdm(df_raw_coords.index):
    bairro = df_raw_coords.loc[i, "bairro"]

    location = geocode(bairro)
    df_raw_coords.loc[i, "latitude"] = location.latitude
    df_raw_coords.loc[i, "longitude"] = location.longitude

df_raw_coords = df_raw_coords.reset_index().drop(columns=["bairro", "cidade"]).rename(columns={"index": "id"})
df_transf_coords = spark.createDataFrame(df_raw_coords)
df_transf_coords.show()

 50%|█████     | 48/96 [00:47<00:45,  1.06it/s]

#### 3. Tipo

In [12]:
df_transf_tipo = df_raw_bueiros.select("Tipo").drop_duplicates().withColumnRenamed("Tipo", "nome")\
    .withColumn("id", F.row_number().over(Window.orderBy(F.lit(1))))

#### 4. Bueiro

In [13]:
df_transf_bueiro = df_raw_bueiros.withColumnRenamed("Extensão (m)", "extensao").withColumnRenamed("Dimensão (m)", "dimensao")\
    .withColumnRenamed("Tipo", "nome")\
    .withColumn("endereco", F.concat(F.col("Rodovia"), F.lit(" KM"), F.col("KM").cast("string")))\
    .withColumn("bairro", get_district("endereco")).select("extensao", "dimensao", "nome", "bairro")

df_transf_bueiro = df_transf_bueiro.join(df_transf_tipo, on="nome").withColumnRenamed("id", "tipo_id").drop("nome")
df_transf_bueiro = df_transf_bueiro.join(df_transf_distritos, on="bairro").withColumnRenamed("id", "endereco_id").drop("bairro", "cidade")

df_transf_bueiro = df_transf_bueiro.withColumn("id", F.row_number().over(Window.orderBy(F.lit(1))))

#### 5. Chuva e solo

In [27]:
OPEN_METEO_BASE_URL = "https://api.open-meteo.com/v1/forecast?latitude={}&longitude={}&hourly=rain&past_days=31&forecast_days=1"
lats_longs_distr = df_transf_coords.drop("id").withColumnRenamed("endereco_id", "id").join(df_transf_distritos, on="id").select("id", "latitude", "longitude")\
    .withColumnRenamed("id", "distrito_id").drop_duplicates().toPandas()

full_hourly_rain = []
chunk_size = 50
for i in tqdm(lats_longs_distr.index):
    distrito_id = lats_longs_distr.loc[i, "distrito_id"]
    lat = lats_longs_distr.loc[i, "latitude"]
    long = lats_longs_distr.loc[i, "longitude"]
    
    res = requests.get(OPEN_METEO_BASE_URL.format(lat, long))
    res.raise_for_status()
    
    subset_hourly_rain = load(BytesIO(res.content))
    subset_hourly_rain["distrito_id"] = distrito_id
    full_hourly_rain.append(subset_hourly_rain)

100%|██████████| 96/96 [00:42<00:00,  2.26it/s]                                 


In [32]:
df_raw_rain = pd.DataFrame.from_records(full_hourly_rain)[["distrito_id", "latitude", "longitude", "hourly"]]
df_raw_rain["time"] = df_raw_rain["hourly"].apply(lambda x: x["time"])
df_raw_rain["rain"] = df_raw_rain["hourly"].apply(lambda x: x["rain"])
df_raw_rain = df_raw_rain.drop(columns="hourly").explode(["time", "rain"])

df_raw_rain = df_raw_rain[df_raw_rain["rain"] > 0]

df_raw_rain["group_id"] = df_raw_rain['rain'].ne(df_raw_rain['rain'].shift()).cumsum()

df_agg_rain = df_raw_rain.groupby(["distrito_id", "latitude", "longitude", "group_id"]).agg(
    dt_hr_inicio=('time', 'first'), dt_hr_fim=('time', 'last'), qtd=('rain', 'first')
).reset_index()

In [ ]:
df_transf_rain = spark.createDataFrame(df_agg_rain).drop("group_id", "latitude", "longitude").withColumn("id", F.row_number().over(Window.orderBy(F.lit(1))))

26/04/21 21:48:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 21:48:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 21:48:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------+----------------+----------------+---+---+
|distrito_id|    dt_hr_inicio|       dt_hr_fim|qtd| id|
+-----------+----------------+----------------+---+---+
|          0|2026-03-31T12:00|2026-04-08T15:00|0.1|  1|
|          1|2026-03-31T12:00|2026-04-08T15:00|0.1|  2|
|          2|2026-03-31T12:00|2026-04-08T15:00|0.1|  3|
|          3|2026-03-31T12:00|2026-04-08T15:00|0.1|  4|
|          4|2026-03-31T12:00|2026-04-08T15:00|0.1|  5|
|          5|2026-03-31T12:00|2026-04-08T15:00|0.1|  6|
|          6|2026-03-31T12:00|2026-04-08T15:00|0.1|  7|
|          7|2026-03-31T12:00|2026-04-08T15:00|0.1|  8|
|          8|2026-03-31T12:00|2026-04-08T15:00|0.1|  9|
|          9|2026-03-31T12:00|2026-04-08T15:00|0.1| 10|
|         10|2026-03-31T12:00|2026-04-08T15:00|0.1| 11|
|         11|2026-03-31T12:00|2026-04-08T15:00|0.1| 12|
|         12|2026-03-31T12:00|2026-04-08T15:00|0.1| 13|
|         13|2026-03-31T12:00|2026-04-08T15:00|0.1| 14|
|         14|2026-03-31T12:00|2026-04-08T15:00|0

26/04/21 21:48:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 21:48:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


#### 6. Complemento coordenada

In [ ]:
df_pandas_coords = df_transf_coords.toPandas().drop(columns="endereco_id").rename(columns={"id":"coordenada_id"})
df_pandas_coords["rua"] = ""
df_pandas_coords["numero"] = ""
df_pandas_coords["cep"] = ""

geolocator = Nominatim(user_agent="tcc")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

for i in tqdm(df_pandas_coords.index):
    lat = df_pandas_coords.loc[i, "latitude"]
    long = df_pandas_coords.loc[i, "longitude"]

    location = reverse((lat, long))
    address = location.raw["address"]

    df_pandas_coords.loc[i, "rua"] = address["road"]
    df_pandas_coords.loc[i, "numero"] = address["house_number"]
    df_pandas_coords.loc[i, "cep"] = address["postcode"].str.replace("-", "")
df_pandas_coords["cep"] = df_pandas_coords["cep"].str.replace("-", "")

100%|██████████| 96/96 [01:35<00:00,  1.01it/s]


In [73]:
df_tranf_compl_coord = spark.createDataFrame(df_pandas_coords.reset_index().drop(columns=["latitude", "longitude"]).rename(columns={"index": "id"}))

## Load

### 1. Distrito

In [75]:
df_transf_distritos.write.jdbc(DB_URL, "distrito", properties=DB_PROPERTIES, mode="append")

### 2. Coordenada

In [76]:
df_transf_coords.write.jdbc(DB_URL, "coordenada", properties=DB_PROPERTIES, mode="append")

26/04/21 22:12:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


### 3. Tipo

In [77]:
df_transf_tipo.write.jdbc(DB_URL, "tipo", properties=DB_PROPERTIES, mode="append")

26/04/21 22:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 2

### 4. Bueiro

In [78]:
df_transf_bueiro.write.jdbc(DB_URL, "bueiro", properties=DB_PROPERTIES, mode="append")

26/04/21 22:12:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 2

### 5. Chuva

In [79]:
df_transf_rain.write.jdbc(DB_URL, "chuva", properties=DB_PROPERTIES, mode="append")

26/04/21 22:12:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


### 6. Complemento coordenadas

In [80]:
df_tranf_compl_coord.write.jdbc(DB_URL, "complemento_coordenada", properties=DB_PROPERTIES, mode="append")